## **Metaadatok kinyerése és duplikációk szűrése**

In [ ]:
import h5py
import json
import os
import numpy as np
from glob import glob

DATASET_DIR = "./Dataset/"
OUTPUT_FILE = os.path.join(DATASET_DIR, "spotify_dataset.h5")

json_files = sorted(glob(os.path.join(DATASET_DIR, "mpd.slice.*.json")))
print(f"{len(json_files)} JSON fájl találva.")

# --- 1. Adatok összegyűjtése ---
tracks_dict = {}
playlists_list = []
playlist_tracks = []

for fpath in json_files:
    print(f"  Feldolgozás: {os.path.basename(fpath)}")
    with open(fpath, "r", encoding="utf-8") as f:
        data = json.load(f)

    for pl in data["playlists"]:
        playlists_list.append({
            "pid":           pl["pid"],
            "name":          pl.get("name", ""),
            "collaborative": pl.get("collaborative", "false") == "true",
            "modified_at":   pl.get("modified_at", 0),
            "num_tracks":    pl.get("num_tracks", 0),
            "num_albums":    pl.get("num_albums", 0),
            "num_followers": pl.get("num_followers", 0),
        })

        for track in pl["tracks"]:
            uri = track["track_uri"]
            if uri not in tracks_dict:
                tracks_dict[uri] = {
                    "track_name":  track.get("track_name", ""),
                    "artist_name": track.get("artist_name", ""),
                    "album_name":  track.get("album_name", ""),
                    "artist_uri":  track.get("artist_uri", ""),
                    "album_uri":   track.get("album_uri", ""),
                    "duration_ms": track.get("duration_ms", 0),
                }

            playlist_tracks.append({
                "pid":       pl["pid"],
                "track_uri": uri,
                "pos":       track["pos"]
            })

print(f"\nEgyedi dalok száma:        {len(tracks_dict):,}")
print(f"Lejátszási listák száma:   {len(playlists_list):,}")
print(f"Playlist–dal kapcsolatok:  {len(playlist_tracks):,}")

# --- 2. HDF5 írás ---
dt_str = h5py.special_dtype(vlen=str)

with h5py.File(OUTPUT_FILE, "w") as hf:

    # ── /tracks ──────────────────────────────────────────────────────────────
    track_uris = list(tracks_dict.keys())

    tracks_grp = hf.create_group("tracks")

    def write_str_ds(grp, name, data_list):
        arr = np.array(data_list, dtype=object)
        grp.create_dataset(name, data=arr, dtype=dt_str)

    write_str_ds(tracks_grp, "track_uri",   track_uris)
    write_str_ds(tracks_grp, "track_name",  [tracks_dict[u]["track_name"]  for u in track_uris])
    write_str_ds(tracks_grp, "artist_name", [tracks_dict[u]["artist_name"] for u in track_uris])
    write_str_ds(tracks_grp, "album_name",  [tracks_dict[u]["album_name"]  for u in track_uris])
    write_str_ds(tracks_grp, "artist_uri",  [tracks_dict[u]["artist_uri"]  for u in track_uris])
    write_str_ds(tracks_grp, "album_uri",   [tracks_dict[u]["album_uri"]   for u in track_uris])
    tracks_grp.create_dataset(
        "duration_ms",
        data=np.array([tracks_dict[u]["duration_ms"] for u in track_uris], dtype=np.int32)
    )

    # ── /playlists ────────────────────────────────────────────────────────────
    pl_grp = hf.create_group("playlists")

    pl_grp.create_dataset(
        "pid",
        data=np.array([p["pid"] for p in playlists_list], dtype=np.int32)
    )
    pl_grp.create_dataset(
        "collaborative",
        data=np.array([p["collaborative"] for p in playlists_list], dtype=np.bool_)
    )
    pl_grp.create_dataset(
        "modified_at",
        data=np.array([p["modified_at"] for p in playlists_list], dtype=np.int64)
    )
    pl_grp.create_dataset(
        "num_tracks",
        data=np.array([p["num_tracks"] for p in playlists_list], dtype=np.int32)
    )
    pl_grp.create_dataset(
        "num_albums",
        data=np.array([p["num_albums"] for p in playlists_list], dtype=np.int32)
    )
    pl_grp.create_dataset(
        "num_followers",
        data=np.array([p["num_followers"] for p in playlists_list], dtype=np.int32)
    )
    write_str_ds(pl_grp, "name", [p["name"] for p in playlists_list])

    # ── /playlist_tracks ──────────────────────────────────────────────────────
    pt_grp = hf.create_group("playlist_tracks")

    pt_grp.create_dataset(
        "pid",
        data=np.array([r["pid"] for r in playlist_tracks], dtype=np.int32)
    )
    write_str_ds(pt_grp, "track_uri", [r["track_uri"] for r in playlist_tracks])
    pt_grp.create_dataset(
        "pos",
        data=np.array([r["pos"] for r in playlist_tracks], dtype=np.int16)
    )

print(f"\n✅ HDF5 mentve: {OUTPUT_FILE}")

# --- 3. Ellenőrzés ---
with h5py.File(OUTPUT_FILE, "r") as hf:
    print(f"\n📊 Ellenőrzés:")
    print(f"  Egyedi dalok:        {hf['tracks/track_uri'].shape[0]:,}")
    print(f"  Lejátszási listák:   {hf['playlists/pid'].shape[0]:,}")
    print(f"  Playlist–dal kapcs.: {hf['playlist_tracks/pid'].shape[0]:,}")

1000 JSON fájl találva.
  Feldolgozás: mpd.slice.0-999.json
  Feldolgozás: mpd.slice.1000-1999.json
  Feldolgozás: mpd.slice.10000-10999.json
  Feldolgozás: mpd.slice.100000-100999.json
  Feldolgozás: mpd.slice.101000-101999.json
  Feldolgozás: mpd.slice.102000-102999.json
  Feldolgozás: mpd.slice.103000-103999.json
  Feldolgozás: mpd.slice.104000-104999.json
  Feldolgozás: mpd.slice.105000-105999.json
  Feldolgozás: mpd.slice.106000-106999.json
  Feldolgozás: mpd.slice.107000-107999.json
  Feldolgozás: mpd.slice.108000-108999.json
  Feldolgozás: mpd.slice.109000-109999.json
  Feldolgozás: mpd.slice.11000-11999.json
  Feldolgozás: mpd.slice.110000-110999.json
  Feldolgozás: mpd.slice.111000-111999.json
  Feldolgozás: mpd.slice.112000-112999.json
  Feldolgozás: mpd.slice.113000-113999.json
  Feldolgozás: mpd.slice.114000-114999.json
  Feldolgozás: mpd.slice.115000-115999.json
  Feldolgozás: mpd.slice.116000-116999.json
  Feldolgozás: mpd.slice.117000-117999.json
  Feldolgozás: mpd.slice

In [ ]:
#Az első lejátszási lista 10 dala
with h5py.File(OUTPUT_FILE, "r") as hf:
    first_pid = hf["playlists/pid"][0]
    print(f"\n🎵 Az első lejátszási lista (PID={first_pid}) első 10 dala:")
    for i in range(10):
        track_uri = hf["playlist_tracks/track_uri"][i]
        track_name = hf["tracks/track_name"][np.where(hf["tracks/track_uri"][:] == track_uri)[0][0]]
        artist_name = hf["tracks/artist_name"][np.where(hf["tracks/track_uri"][:] == track_uri)[0][0]]
        print(f"  {i+1}. {track_name} - {artist_name}")


🎵 Az első lejátszási lista (PID=0) első 10 dala:
  1. b'Lose Control (feat. Ciara & Fat Man Scoop)' - b'Missy Elliott'
  2. b'Toxic' - b'Britney Spears'
  3. b'Crazy In Love' - b'Beyonc\xc3\xa9'
  4. b'Rock Your Body' - b'Justin Timberlake'
  5. b"It Wasn't Me" - b'Shaggy'
  6. b'Yeah!' - b'Usher'
  7. b'My Boo' - b'Usher'
  8. b'Buttons' - b'The Pussycat Dolls'
  9. b'Say My Name' - b"Destiny's Child"
  10. b'Hey Ya! - Radio Mix / Club Mix' - b'OutKast'


In [ ]:
# Néhány dal kiírása ellenőrzésképpen
import h5py

OUTPUT_FILE = os.path.join(DATASET_DIR, "spotify_dataset.h5")

with h5py.File(OUTPUT_FILE, "r") as hf:
    n_tracks    = hf["tracks/track_uri"].shape[0]
    n_playlists = hf["playlists/pid"].shape[0]
    n_pt        = hf["playlist_tracks/pid"].shape[0]

    print(f"Egyedi dalok száma:       {n_tracks:,}")
    print(f"Lejátszási listák száma:  {n_playlists:,}")
    print(f"Playlist–dal kapcsolatok: {n_pt:,}")

    # Első néhány dal kiírása ellenőrzésképpen
    print("\nElső 5 dal:")
    for i in range(min(5, n_tracks)):
        uri  = hf["tracks/track_uri"][i].decode()
        name = hf["tracks/track_name"][i].decode()
        artist = hf["tracks/artist_name"][i].decode()
        print(f"  [{i}] {artist} – {name}  ({uri})")

Egyedi dalok száma:       2,262,292
Lejátszási listák száma:  1,000,000
Playlist–dal kapcsolatok: 66,346,428

Első 5 dal:
  [0] Missy Elliott – Lose Control (feat. Ciara & Fat Man Scoop)  (spotify:track:0UaMYEvWZi0ZqiDOoHU3YI)
  [1] Britney Spears – Toxic  (spotify:track:6I9VzXrHxO9rA9A5euc8Ak)
  [2] Beyoncé – Crazy In Love  (spotify:track:0WqIKmW4BTrj3eJFmnCKMv)
  [3] Justin Timberlake – Rock Your Body  (spotify:track:1AWQoqb9bSvzTjaLralEkT)
  [4] Shaggy – It Wasn't Me  (spotify:track:1lzr43nnXAijIGYnCT8M8H)


# **Zeneszámok letöltése, spektrogrammá alakítása és eltárolása**

## Spotify API

In [ ]:
#Zeneszámok letöltése ideiglenesen

import time
import h5py
import subprocess
import os

DATASET_DIR = "/content/drive/MyDrive/Diplomamunka/Dataset/"
OUTPUT_FILE = os.path.join(DATASET_DIR, "spotify_mpd.h5")
JS_SCRIPT_PATH = os.path.join(DATASET_DIR, "enhanced_search.js")

print("Adatok beolvasása a HDF5 fájlból...\n")
with h5py.File(OUTPUT_FILE, "r") as hf:
    track_names = [name.decode('utf-8') for name in hf['tracks/track_name'][:3]]
    artist_names = [artist.decode('utf-8') for artist in hf['tracks/artist_name'][:3]]

# 2. Iteráljunk végig ezen az 1600 dalon és hívjuk meg a Node.js szkriptet
for i in range(len(track_names)):
    t_name = track_names[i]
    a_name = artist_names[i]

    process_result = subprocess.run(
        ['node', JS_SCRIPT_PATH, t_name, a_name], # <--- Itt a változás!
        capture_output=True,
        text=True
    )

    # Kiírjuk a Node.js szkript eredményét (amit a console.log-ok generáltak)
    if process_result.returncode == 0:
        print(f"[{2890+i}/3600] {process_result.stdout}")
    else:
        print("❌ Hiba történt a Node.js futtatása közben:\n", process_result.stderr)

    #Várunk egy kicsit a következő iteráció előtt, hogy ne terheljük túl a Spotify API-t
    time.sleep(2)

print("✅ Kész! Az 2890-3600. dal feldolgozása befejeződött.")

Adatok beolvasása a HDF5 fájlból...

[2890/3600] [dotenv@17.3.1] injecting env (2) from .env -- tip: 🛠️  run anywhere with `dotenvx run -- yourcommand`

🔍 Keresés: Lose Control (feat. Ciara & Fat Man Scoop) - Missy Elliott...
❌ Nem található a dal a Spotify-on.

[2891/3600] [dotenv@17.3.1] injecting env (2) from .env -- tip: 🤖 agentic secret storage: https://dotenvx.com/as2

🔍 Keresés: Toxic - Britney Spears...
❌ Nem található a dal a Spotify-on.

[2892/3600] [dotenv@17.3.1] injecting env (2) from .env -- tip: ⚙️  override existing env vars with { override: true }

🔍 Keresés: Crazy In Love - Beyoncé...
❌ Nem található a dal a Spotify-on.

✅ Kész! Az 2890-3600. dal feldolgozása befejeződött.


## Youtube API

In [ ]:
import time
import h5py
import subprocess
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

DATASET_DIR = "./Dataset/"
OUTPUT_FILE = os.path.join(DATASET_DIR, "spotify_dataset.h5")
MP3_DIR = "./MP3/"
os.makedirs(MP3_DIR, exist_ok=True)

with h5py.File(OUTPUT_FILE, "r") as hf:
    track_names = [name.decode('utf-8') for name in hf['tracks/track_name'][28000:30000]]
    artist_names = [artist.decode('utf-8') for artist in hf['tracks/artist_name'][28000:30000]]

def download_track(args):
    i, t_name, a_name = args
    safe_name = f"{t_name} - {a_name}".replace('/', '-').replace('\\', '-').replace(':', '-').replace('*', '-').replace('?', '-').replace('"', '-').replace('<', '-').replace('>', '-').replace('|', '-')
    output_path = os.path.join(MP3_DIR, f"{safe_name}.mp3")

    if os.path.exists(output_path):
        return i, t_name, a_name, "skip"

    queries = [
        f"{t_name} {a_name} official audio",
        f"{t_name} {a_name}",
        f"{t_name} {a_name} official video",
        f"{t_name} {a_name} lyrics",
    ]

    for query in queries:
        cmd = [
            'yt-dlp',
            f'ytsearch1:{query}',
            '--extract-audio',
            '--audio-format', 'mp3',
            '--audio-quality', '0',
            '--download-sections', '*35-65',
            '--force-keyframes-at-cuts',
            '-o', output_path,
            '--quiet', '--no-warnings'
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0 and os.path.exists(output_path):
            return i, t_name, a_name, "ok"

    return i, t_name, a_name, "fail"

tasks = [(i, track_names[i], artist_names[i]) for i in range(len(track_names))]

with ThreadPoolExecutor(max_workers=15) as executor:  # 10 párhuzamos letöltés
    futures = {executor.submit(download_track, task): task for task in tasks}
    for future in as_completed(futures):
        i, t_name, a_name, status = future.result()
        if status == "ok":
            print(f"[{28000+i+1}/30000] ✅ {t_name} - {a_name}")
        elif status == "skip":
            print(f"[{28000+i+1}/30000] ⏭️ Már létezik: {t_name} - {a_name}")
        else:
            print(f"[{28000+i+1}/30000] ❌ Sikertelen: {t_name} - {a_name}")

print("A 28001-30000. dalok letöltése befejeződött.")

[28008/30000] ✅ Veinte Anos - Buena Vista Social Club
[28003/30000] ✅ I Wish You Would - Taylor Swift
[28012/30000] ✅ Orgullecida - Buena Vista Social Club
[28014/30000] ✅ Buena Vista Social Club - Buena Vista Social Club
[28015/30000] ✅ La Bayamesa - Buena Vista Social Club
[28010/30000] ✅ Candela - Buena Vista Social Club
[28005/30000] ✅ Pueblo Nuevo - Buena Vista Social Club
[28006/30000] ✅ Dos Gardenias - Buena Vista Social Club
[28004/30000] ✅ El Cuarto De Tula - Buena Vista Social Club
[28011/30000] ✅ Amor De Loca Juventud - Buena Vista Social Club
[28009/30000] ✅ El Carretero - Buena Vista Social Club
[28007/30000] ✅ Y tú qué has hecho - Compay Segundo
[28013/30000] ✅ Murmullo - Buena Vista Social Club
[28016/30000] ✅ Bruca Manigua - Ibrahim Ferrer
[28017/30000] ✅ Herido De Sombras - Ibrahim Ferrer
[28022/30000] ✅ Cienfuegos Tiene Su Guaguancó - Ibrahim Ferrer
[28021/30000] ✅ Nuestra Ultima Cita - Ibrahim Ferrer
[28018/30000] ✅ Marieta - Ibrahim Ferrer
[28024/30000] ✅ Aquellos O

# **HDF5 létrehozása, feltöltése, lekérése**

## HDF5 kiegészítése

In [ ]:
#Bővíthető HDF5 struktúra létrehozása a Spotify API-ból származó adatok tárolásához

import h5py
import numpy as np
import os

DATASET_DIR = "./Dataset/"
OUTPUT_FILE = os.path.join(DATASET_DIR, "spotify_dataset.h5")

with h5py.File(OUTPUT_FILE, "a") as hf:

    # --- SPECTROGRAMS GROUP ---
    if "spectrograms" not in hf:
        spec_grp = hf.create_group("spectrograms")
        print("✅ '/spectrograms' group létrehozva.")
    else:
        spec_grp = hf["spectrograms"]
        print("ℹ️ '/spectrograms' group már létezik.")

    if "mel" not in spec_grp:
        spec_grp.create_dataset("mel", shape=(0, 128, 1280), maxshape=(None, 128, 1280),
                                dtype=np.float32, chunks=(1, 128, 1280), compression="lzf")
        print("✅ 'mel' dataset létrehozva.")

    if "chroma" not in spec_grp:
        spec_grp.create_dataset("chroma", shape=(0, 12, 1280), maxshape=(None, 12, 1280),
                                dtype=np.float32, chunks=(1, 12, 1280), compression="lzf")
        print("✅ 'chroma' dataset létrehozva.")

    if "tempogram" not in spec_grp:
        spec_grp.create_dataset("tempogram", shape=(0, 384, 1280), maxshape=(None, 384, 1280),
                                dtype=np.float32, chunks=(1, 384, 1280), compression="lzf")
        print("✅ 'tempogram' dataset létrehozva.")

    # --- ML GROUP ---
    if "ml" not in hf:
        ml_grp = hf.create_group("ml")
        print("✅ '/ml' group létrehozva.")
    else:
        ml_grp = hf["ml"]
        print("ℹ️ '/ml' group már létezik.")

    dt_str = h5py.special_dtype(vlen=str)

    if "track_uri" not in ml_grp:
        ml_grp.create_dataset("track_uri", shape=(0,), maxshape=(None,), dtype=dt_str)
        print("✅ 'track_uri' dataset létrehozva.")

    if "split" not in ml_grp:
        ml_grp.create_dataset("split", shape=(0,), maxshape=(None,), dtype=dt_str)
        print("✅ 'split' dataset létrehozva.")

print("\n🎉 HDF5 struktúra készen áll!")

✅ '/spectrograms' group létrehozva.
✅ 'mel' dataset létrehozva.
✅ 'chroma' dataset létrehozva.
✅ 'tempogram' dataset létrehozva.
✅ '/ml' group létrehozva.
✅ 'track_uri' dataset létrehozva.
✅ 'split' dataset létrehozva.

🎉 HDF5 struktúra készen áll!


# MP3 fájlok feltöltése Drive-ra

In [ ]:
#Mennyi dalt kellene eltárolni check
import h5py
import numpy as np

DATASET_DIR = "/content/drive/MyDrive/Diplomamunka/Dataset/"
OUTPUT_FILE = os.path.join(DATASET_DIR, "spotify_mpd.h5")

with h5py.File(OUTPUT_FILE, "r") as hf:
    pids = hf['playlist_tracks/pid'][:]
    uris = hf['playlist_tracks/track_uri'][:]
    all_pids = hf['playlists/pid'][:]

    # Első 1000 playlist pid-jei
    first_1000_pids = all_pids[:1500]

    # Ezekben szereplő egyedi dalok
    mask = np.isin(pids, first_1000_pids)
    unique_tracks = np.unique(uris[mask])

    print(f"Első 1000 playlist egyedi daljai: {len(unique_tracks):,}")

Első 1000 playlist egyedi daljai: 47,023


## MP3 fájlok másolása helyi könyvtárból Drive-ra

In [ ]:
import shutil
import os
from glob import glob

DRIVE_MP3_DIR = "/content/drive/MyDrive/Diplomamunka/MP3/"

# Mappa létrehozása ha még nincs
os.makedirs(DRIVE_MP3_DIR, exist_ok=True)

# MP3 fájlok keresése a gyökérben
mp3_files = glob("/content/MP3/*.mp3")
print(f"{len(mp3_files)} MP3 fájl található.")

for i, mp3_path in enumerate(mp3_files):
    filename = os.path.basename(mp3_path)
    dest = os.path.join(DRIVE_MP3_DIR, filename)
    shutil.copy(mp3_path, dest)
    print(f"[{i+1}/{len(mp3_files)}] ✅ {filename}")

print(f"\n🎉 Kész! {len(mp3_files)} fájl átmásolva ide: {DRIVE_MP3_DIR}")

1197 MP3 fájl található.
[1/1197] ✅ Symphony No. 6 in B minor (-Pathétique-), Op. 74 - Scherzo - All-Union Radio Symphony Orchestra.mp3
[2/1197] ✅ Goddamn - Falling In Reverse.mp3
[3/1197] ✅ Tidal Wave - EVVY.mp3
[4/1197] ✅ Vocalise Op.34-14 - South German Philharmonic Orchestra.mp3
[5/1197] ✅ The One I Love - R.E.M..mp3
[6/1197] ✅ Swan Lake, Ballet suite, op. 20a- I. Scene - Lake in the Moonlight - - Symphonic Festival Orchestra.mp3
[7/1197] ✅ Love This Life - T.I..mp3
[8/1197] ✅ Brandenburg Concerto No.3 In G Major, 1st Movement - Philharmonia Slavonica.mp3
[9/1197] ✅ I'm Different - 2 Chainz.mp3
[10/1197] ✅ Somebody To You - The Vamps.mp3
[11/1197] ✅ Iron Sky - Paolo Nutini.mp3
[12/1197] ✅ Niggaz Know - J. Cole.mp3
[13/1197] ✅ Sorry - Buckcherry.mp3
[14/1197] ✅ New Love - Dua Lipa.mp3
[15/1197] ✅ Blast - Clams Casino.mp3
[16/1197] ✅ RIP Roach - XXXTENTACION.mp3
[17/1197] ✅ A Heart Arcane - Horse Feathers.mp3
[18/1197] ✅ Colour My Eyes - Mark Norman.mp3
[19/1197] ✅ I Can't Keep Up - 

## MP3 fájlok törlése a helyi mappából

In [ ]:
import os
import glob

# Colab lokális MP3-ak
mp3_files = glob.glob("/content/MP3/*.mp3")
print(f"Lokális MP3-ak száma: {len(mp3_files)}")

# Ha rendben van a szám, töröld őket:
for f in mp3_files:
    os.remove(f)
print("✅ Törölve!")

Lokális MP3-ak száma: 1994
✅ Törölve!


In [ ]:
import os

MP3_DIR = "/content/drive/MyDrive/Diplomamunka/MP3"
files = [f for f in os.listdir(MP3_DIR) if f.endswith('.mp3')]
print(f"MP3 fájlok száma: {len(files)}")

MP3 fájlok száma: 17847


# Hiányzó zenék és duplikátumok ellenőrzése

In [ ]:
#Hiányzó zenék ellenőrzése a HDF5 alapján
import os
import h5py
import re
import unicodedata

DATASET_DIR = "/content/drive/MyDrive/Diplomamunka/Dataset/"
OUTPUT_FILE = os.path.join(DATASET_DIR, "spotify_mpd.h5")
MP3_DIR = "/content/drive/MyDrive/Diplomamunka/MP3"

def get_safe_filename(song_name, artist_name):
    base = f"{song_name} - {artist_name}"
    return re.sub(r'[/\\?%*:|"<>]', '-', base) + ".mp3"

def normalize(s):
    return unicodedata.normalize('NFC', s)

# Meglévő MP3-ak normalizálva
existing_mp3s = set(normalize(f) for f in os.listdir(MP3_DIR))
print(f"MP3 mappában: {len(existing_mp3s)} fájl")

with h5py.File(OUTPUT_FILE, "r") as hf:
    track_names  = [n.decode() for n in hf['tracks/track_name'][:14000]]
    artist_names = [a.decode() for a in hf['tracks/artist_name'][:14000]]

missing = []
for i, (t, a) in enumerate(zip(track_names, artist_names)):
    filename = normalize(get_safe_filename(t, a))
    if filename not in existing_mp3s:
        missing.append((i+1, t, a))

print(f"Hiányzó MP3-ak száma: {len(missing)}")
for idx, t, a in missing:
    print(f"  [{idx}] {t} - {a}")

MP3 mappában: 13911 fájl
Hiányzó MP3-ak száma: 40
  [1403] Degas Park - Kevin Abstract
  [1531] ...And Ever - Titus Andronicus
  [3127] ... Even Though You're With Another Girl - Trentemøller
  [3219] Time of Our Lives - Main Title Theme From "I Didn't Do It" - Olivia Holt
  [3377] Fuck This Shit I'm Out - The Theme Song
  [4118] Sovereign - Sweet Rain Records Master - Daryl Coley
  [4252] Agar Tum Saath Ho - Prokumar & Arijit Singh & Alka Yagnik
  [4623] In Summer - Josh Gad
  [5241] Winnie The Pooh - From "Winnie the Pooh and the Honey Tree" - Disney Studio Chorus
  [6191] The Water Buffalo Song - VeggieTales
  [6245] Bonus Track - Da Vinci's Notebook
  [6384] I Put A Spell On You - Silver Screen Superstars
  [6454] Laffy Taffy - Explicit Album Version w/sample/APPROVED-FINAL - D4L
  [6815] Where You Are - Christopher Jackson
  [7409] Adder(F)All - Tei Shi
  [7937] Let It Go - From "Frozen"/Soundtrack Version - Idina Menzel
  [8430] PUSSY - Rammstein
  [9348] I Believe In Jesus - Nor

In [ ]:
#Duplikátumok ellenőrzése a fájlnevek alapján
with h5py.File(OUTPUT_FILE, "r") as hf:
    track_names  = [n.decode() for n in hf['tracks/track_name'][:2989]]
    artist_names = [a.decode() for a in hf['tracks/artist_name'][:2989]]

# Fájlnevek generálása
filenames = [normalize(get_safe_filename(t, a)) for t, a in zip(track_names, artist_names)]

# Duplikátumok keresése
from collections import Counter
counts = Counter(filenames)
duplicates = {f: c for f, c in counts.items() if c > 1}

print(f"Duplikált fájlnevek száma: {len(duplicates)}")
for f, c in duplicates.items():
    print(f"  {c}x: {f}")

Duplikált fájlnevek száma: 4
  2x: All Time Low - Jon Bellion.mp3
  2x: Cheap Thrills - Sia.mp3
  2x: Panda - Desiigner.mp3
  2x: Need You Now - Lady Antebellum.mp3


In [ ]:
import h5py
import json
import os

# --- ELÉRÉSI UTAK (Módosítsd a Ryzen gépednek megfelelően!) ---
HDF5_FILE = "./Dataset/spotify_dataset.h5"  # A te nagy HDF5 fájlod
MAPPING_SAVE_PATH = "./mp3_to_uri_mapping.json"

mp3_to_uri = {}

print("HDF5 fájl olvasása és a mesterszótár építése...")
with h5py.File(HDF5_FILE, "r") as hf:
    # Lekérjük, hány egyedi dal van összesen a fájlban (kb. 2.2 millió lesz)
    total_tracks = hf['tracks/track_uri'].shape[0]
    print(f"Összesen {total_tracks:,} dal feldolgozása a memóriában...\n")

    # Chunkos (darabolt) olvasás, hogy villámgyors legyen az SSD-ről
    chunk_size = 500_000
    for start in range(0, total_tracks, chunk_size):
        end = min(start + chunk_size, total_tracks)

        # Betöltünk félmillió nevet és URI-t
        t_names = hf['tracks/track_name'][start:end]
        a_names = hf['tracks/artist_name'][start:end]
        uris = hf['tracks/track_uri'][start:end]

        for t_name_b, a_name_b, uri_b in zip(t_names, a_names, uris):
            # HDF5 byte dekódolás stringgé
            t_name = t_name_b.decode('utf-8') if isinstance(t_name_b, bytes) else str(t_name_b)
            a_name = a_name_b.decode('utf-8') if isinstance(a_name_b, bytes) else str(a_name_b)
            uri = uri_b.decode('utf-8') if isinstance(uri_b, bytes) else str(uri_b)

            # PONTOSAN a te tisztító logikád a letöltő szkriptből!
            safe_name = f"{t_name} - {a_name}".replace('/', '-').replace('\\', '-').replace(':', '-').replace('*', '-').replace('?', '-').replace('"', '-').replace('<', '-').replace('>', '-').replace('|', '-')
            filename = f"{safe_name}.mp3"

            # Hozzáadjuk a szótárhoz
            mp3_to_uri[filename] = uri

        print(f"  [{end:,} / {total_tracks:,}] feldolgozva...")

print(f"\nSzótár elkészült! {len(mp3_to_uri):,} egyedi fájlnév-URI párosítás memóriában.")

# Elmentjük JSON-ba
with open(MAPPING_SAVE_PATH, "w", encoding="utf-8") as f:
    json.dump(mp3_to_uri, f, indent=4, ensure_ascii=False)

print(f"✅ MAPPING fájl sikeresen elmentve: {MAPPING_SAVE_PATH}")

HDF5 fájl olvasása és a mesterszótár építése...
Összesen 2,262,292 dal feldolgozása a memóriában...

  [500,000 / 2,262,292] feldolgozva...
  [1,000,000 / 2,262,292] feldolgozva...
  [1,500,000 / 2,262,292] feldolgozva...
  [2,000,000 / 2,262,292] feldolgozva...
  [2,262,292 / 2,262,292] feldolgozva...

Szótár elkészült! 2,189,672 egyedi fájlnév-URI párosítás memóriában.
✅ MAPPING fájl sikeresen elmentve: ./mp3_to_uri_mapping.json


In [ ]:
import os
import json
from gensim.models import Word2Vec

# --- ELÉRÉSI UTAK (Módosítsd a sajátjaidra!) ---
MP3_DIR = "./MP3/"
MODEL_PATH = "./song2vec.model"
MAPPING_FILE = "./mp3_to_uri_mapping.json"
OUTPUT_JSON = "./valid_cnn_dataset.json"

print("1. Word2Vec modell betöltése (Ez eltarthat pici ideig)...")
model = Word2Vec.load(MODEL_PATH)
print(f"   Modell szótár mérete: {len(model.wv):,} egyedi dal.")

print("2. URI szótár betöltése...")
with open(MAPPING_FILE, "r", encoding="utf-8") as f:
    mp3_to_uri = json.load(f)

print("3. Helyi MP3 fájlok beolvasása...")
local_files = [f for f in os.listdir(MP3_DIR) if f.endswith('.mp3')]
print(f"   {len(local_files):,} darab MP3 fájl található a mappában.")

# --- SZŰRÉS ---
valid_data = {}
missing_from_mapping = 0
missing_from_w2v = 0

print("\nÁtfedés vizsgálata (Aposztróf-mentő üzemmódban)...")
for filename in local_files:
    # 1. Alapból keressük a fájlt, ahogy a lemezen van
    if filename in mp3_to_uri:
        uri = mp3_to_uri[filename]
    else:
        # TRÜKK: Ha nem találja, próbáljuk meg "visszaalakítani"
        # az alsóvonalat aposztróffá a kereséshez
        alt_name = filename.replace('_', "'")
        if alt_name in mp3_to_uri:
            uri = mp3_to_uri[alt_name]
        else:
            missing_from_mapping += 1
            continue

    # 2. Megvan az URI, nézzük a Word2Vec-et
    if uri in model.wv:
        valid_data[filename] = uri
    else:
        missing_from_w2v += 1

# --- EREDMÉNYEK KIÍRÁSA ÉS MENTÉSE ---
print("\n" + "="*40)
print("📊 STATISZTIKA:")
print(f"Összes fizikai MP3:         {len(local_files):,}")
print(f"Nincs hozzá URI párosítás:  {missing_from_mapping:,} (pl. letöltési/név hiba)")
print(f"Kiesett a Word2Vec-ből:     {missing_from_w2v:,} (Túl ritka dalok)")
print("-"*40)
print(f"✅ VÉGLEGES, HASZNÁLHATÓ ADAT: {len(valid_data):,} darab dal")
print("="*40)

# Elmentjük az Arany Listát
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(valid_data, f, indent=4, ensure_ascii=False)

print(f"\nA tiszta adatbázis elmentve ide: {OUTPUT_JSON}")

1. Word2Vec modell betöltése (Ez eltarthat pici ideig)...
   Modell szótár mérete: 372,433 egyedi dal.
2. URI szótár betöltése...
3. Helyi MP3 fájlok beolvasása...
   31,607 darab MP3 fájl található a mappában.

Átfedés vizsgálata (Aposztróf-mentő üzemmódban)...

📊 STATISZTIKA:
Összes fizikai MP3:         31,607
Nincs hozzá URI párosítás:  20 (pl. letöltési/név hiba)
Kiesett a Word2Vec-ből:     4,534 (Túl ritka dalok)
----------------------------------------
✅ VÉGLEGES, HASZNÁLHATÓ ADAT: 27,053 darab dal

A tiszta adatbázis elmentve ide: ./valid_cnn_dataset.json


In [2]:
import pickle
import numpy as np

PATH = "../Models/cnn_vectors.pkl" # vagy ahol tárolod

with open(PATH, 'rb') as f:
    data = pickle.load(f)

# 1. Alapadatok
print(f"📊 Összes feldolgozott dal száma: {len(data)}")

# 2. Vegyünk egy mintát (az első kulcsot)
sample_uri = list(data.keys())[0]
sample_vec = data[sample_uri]

print(f"🔍 Minta URI: {sample_uri}")
print(f"📐 Vektor dimenzió: {sample_vec.shape}") # (400,) kell legyen
print(f"🔢 Adattípus: {sample_vec.dtype}")      # float32 az ideális

📊 Összes feldolgozott dal száma: 25510
🔍 Minta URI: spotify:track:7mG2RbhyzGsjpQOz568d39
📐 Vektor dimenzió: (400,)
🔢 Adattípus: float32


In [ ]:
# Mennyi dal van a hdf5 ml/track_uri datasetjében?
import h5py

H5_PATH = "../Dataset/spotify_dataset_compressed.h5"
with h5py.File(H5_PATH, "r") as hf:
    track_uris = hf["ml/track_uri"][:]
    print(f"📊 'ml/track_uri' dataset mérete: {len(track_uris):,} darab URI")

# Mennyi dalhoz tartozik spektrgoram a hdf5-ben?
with h5py.File(H5_PATH, "r") as hf:
    mel_shape = hf["spectrograms/mel"].shape
    print(f"📊 'spectrograms/mel' dataset mérete: {mel_shape[0]:,} darab dal, 128x1280-as spektrogramokkal")

📊 'ml/track_uri' dataset mérete: 27,052 darab URI
📊 'spectrograms/mel' dataset mérete: 27,052 darab dal, 128x1280-as spektrogramokkal


In [ ]:
import os
import h5py
import pickle
import numpy as np
from tqdm import tqdm
import tensorflow as tf

# --- ÚTVONALAK ---
H5_PATH = "/content/drive/MyDrive/Diplomamunka/spotify_dataset_compressed.h5"
CNN_MODEL_PATH = "/content/drive/MyDrive/Diplomamunka/spotify_cnn_model.keras"
CNN_DICT_PATH = "/content/drive/MyDrive/cnn_vectors.pkl"

# --- EGYÉDI OBJEKTUMOK (A betöltéshez kellenek) ---
def cosine_loss(y_true, y_pred):
    return tf.keras.losses.cosine_similarity(y_true, y_pred)

class L2NormLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(L2NormLayer, self).__init__(**kwargs)
    def call(self, inputs):
        return tf.nn.l2_normalize(inputs, axis=-1)

# 1. Modell betöltése
print("🧠 CNN Modell betöltése...")
cnn_model = tf.keras.models.load_model(
    CNN_MODEL_PATH,
    custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
)

cnn_vectors_dict = {}

# 2. Feldolgozás
with h5py.File(H5_PATH, "r") as hf:
    uris = hf["ml/track_uri"][:]
    total_tracks = len(uris)

    print(f"📊 HDF5-ben talált dalok: {total_tracks}")

    for i in tqdm(range(total_tracks), desc="Vektorok kinyerése"):
        try:
            # Spektrogramok beolvasása (Chroma és Tempo is!)
            m = hf["spectrograms/mel"][i]

            # Ellenőrizzük, hogy nem csupa nulla-e (ez szűri ki az 1 hibás dalt)
            if np.all(m == 0):
                continue

            c = hf["spectrograms/chroma"][i]
            t = hf["spectrograms/tempogram"][i]

            # Dimenziók beállítása a modellnek (1, H, W, 1)
            m_in = m[np.newaxis, ..., np.newaxis].astype(np.float32)
            c_in = c[np.newaxis, ..., np.newaxis].astype(np.float32)
            t_in = t[np.newaxis, ..., np.newaxis].astype(np.float32)

            # Predikció
            vec = cnn_model.predict([m_in, c_in, t_in], verbose=0)[0]

            # URI dekódolása és mentése
            uri = uris[i].decode('utf-8') if isinstance(uris[i], bytes) else uris[i]
            cnn_vectors_dict[uri] = vec

            # Időnkénti memóriaürítés a TF-nek
            if i % 1000 == 0:
                tf.keras.backend.clear_session()

        except Exception as e:
            print(f"\n⚠️ Hiba az indexnél {i}: {e}")
            continue

# 3. Mentés
print(f"\n💾 Mentés ide: {CNN_DICT_PATH}")
with open(CNN_DICT_PATH, 'wb') as f:
    pickle.dump(cnn_vectors_dict, f)

print(f"✅ KÉSZ! Összesen {len(cnn_vectors_dict)} valid vektor elmentve.")

🧠 CNN Modell betöltése...
📊 HDF5-ben talált dalok: 27052


Vektorok kinyerése: 100%|██████████| 27052/27052 [1:02:05<00:00,  7.26it/s]



💾 Mentés ide: /content/drive/MyDrive/cnn_vectors.pkl
✅ KÉSZ! Összesen 25510 valid vektor elmentve.


In [6]:
import h5py

H5_PATH = "../Dataset/spotify_dataset_compressed.h5"

with h5py.File(H5_PATH, "r") as hf:
    uris = hf["ml/track_uri"][:]
    total_tracks = len(uris)

    print(len(set(uris)))

25511
